# Train Prior

In [1]:
from maskllm.maskllm import MaskedLinear, MaskedLinearFrozen
from maskllm.sparsegpt import SparseGPT
import timm

/work/hdd/bfxa/dshah13/mac_pruning_fv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_name = 'deit_tiny_patch16_224.fb_in1k'
model = timm.create_model('deit_tiny_patch16_224.fb_in1k', pretrained=True, num_classes=10)

In [8]:
def count_parameters(model):
    """
    Counts the number of trainable parameters in a PyTorch model.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
count_parameters(model)

5526346

In [9]:
import torch
import torch_pruning as tp

def print_model_mac(model):
    example_inputs = (torch.randn(1, 3, 224, 224).to(next(model.parameters()).device),)
    total_macs, _ = tp.utils.count_ops_and_params(model, example_inputs)
    print(f"[📊] Total model MACs: {total_macs/1e9:.3f}G")

In [ ]:
print_model_mac(model)

[📊] Total model MACs: 1.259G


In [ ]:
from utils.analysis_isomorphism import ViTIsomorphicAnalyzer

isomorphic_analyser = ViTIsomorphicAnalyzer(model)
groups = isomorphic_analyser.create_isomorphic_groups(
    target_macs=0.6*1e9,
    baseline_macs=1.25*1e9
    )

[🔍] Found 12 transformer blocks
[📊] Total model MACs: 1.259G
[📊] Target MACs: 0.600G
[📊] Target ratio: 0.520
[🔍] Found 24 MLP layers (3,538,944 total params)
[📊] MLP MAC fraction estimated: 0.666 (3,538,944/5,310,336 params)
[🔍] Found 24 attention layers (1,769,472 total params)
[📊] Attention MAC fraction estimated: 0.333 (1,769,472/5,310,336 params)
[🪫] Max feasible reduction under caps: 0.990 → best achievable ≈ 0.013G
[🎯] LLM suggests: mlp_pruning=0.400, attn_pruning=0.150
[🎯] Using as pruning ratios: r_mlp=0.400, r_attn=0.150
[🎯] Predicted achievement: 0.317 (target: 0.520)
[⚖️] Post-alloc solve -> r_mlp=0.400, r_attn=0.150 (mlp_frac=0.667, attn_frac=0.333, caps: 0.99/0.99)
[📊] MLP: 0.839G → 0.503G (ratio: 0.400)
[📊] Attention: 0.420G → 0.357G (ratio: 0.150)
[📊] Final group summary:
  - MLP blocks: 12 couples
  - Attention blocks: 12 couples
  - Output projections: 0 layers


In [ ]:
from pprint import pprint

pprint(groups)

{'attention_blocks': IsomorphicGroup(name='Attention Blocks (Coupled)',
                                     layers=[AttentionCouple(qkv=Linear(in_features=192, out_features=576, bias=True),
                                                             proj=Linear(in_features=192, out_features=192, bias=True),
                                                             qkv_name='blocks.0.attn.qkv',
                                                             proj_name='blocks.0.attn.proj'),
                                             AttentionCouple(qkv=Linear(in_features=192, out_features=576, bias=True),
                                                             proj=Linear(in_features=192, out_features=192, bias=True),
                                                             qkv_name='blocks.1.attn.qkv',
                                                             proj_name='blocks.1.attn.proj'),
                                             AttentionCouple(qkv=Linear(in_featu

In [ ]:
from data.loaders import get_cifar10_loaders_pbench

batch_size = 1

train_loader, val_loader = get_cifar10_loaders_pbench(batch_size, num_workers=16)

100%|██████████| 170M/170M [00:03<00:00, 54.6MB/s] 
/work/hdd/bfxa/dshah13/mac_pruning_fv/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
/work/hdd/bfxa/dshah13/mac_pruning_fv/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [11]:
import copy

original_model = copy.deepcopy(model)

In [12]:
pprint(original_model)

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)


In [10]:
import torch
import torch.nn as nn

def replace_linear_with_(model, new_class, exclude=[], groups=None, **kwargs):
    """Replace linear layers with new_class in a model. It's an inplace operation.
    
    Args:
        model: The model to modify
        new_class: The class to replace Linear layers with (MaskedLinear)
        exclude_names: List of module names to exclude from replacement
        groups: Dictionary of IsomorphicGroup objects with pruning ratios
        **kwargs: Additional arguments for new_class
    """
    # Default N:M pattern if no groups specified
    default_N = kwargs.get('N', 2)
    default_M = kwargs.get('M', 4)
    
    def get_sparsity_config(full_name):
        """Determine N:M pattern based on full layer name"""
        if groups is None:
            return {'N': default_N, 'M': default_M}
        
        # Extract layer type from full name
        # Example: 'blocks.0.attn.qkv' or 'blocks.0.mlp.fc1'
        name_parts = full_name.split('.')
        
        # Map to group based on structure
        if len(name_parts) >= 4:
            if 'attn' in name_parts:
                if 'qkv' in name_parts or 'proj' in name_parts:
                    # Attention layers (qkv or proj)
                    if hasattr(groups, 'attention_blocks'):
                        ratio = groups.attention_blocks.pruning_ratio
                    elif 'attention_blocks' in groups:
                        ratio = groups['attention_blocks'].pruning_ratio
                    else:
                        ratio = 0.15  # Default for attention
                    return ratio_to_nm(ratio)
            elif 'mlp' in name_parts:
                if 'fc1' in name_parts or 'fc2' in name_parts:
                    # MLP layers
                    if hasattr(groups, 'mlp_blocks'):
                        ratio = groups.mlp_blocks.pruning_ratio
                    elif 'mlp_blocks' in groups:
                        ratio = groups['mlp_blocks'].pruning_ratio
                    else:
                        ratio = 0.4  # Default for MLP
                    return ratio_to_nm(ratio)
        
        # Output projections (head, etc.)
        if 'head' in full_name or 'fc' == name_parts[-1]:
            if hasattr(groups, 'output_projections'):
                ratio = groups.output_projections.pruning_ratio
            elif 'output_projections' in groups:
                ratio = groups['output_projections'].pruning_ratio
            else:
                ratio = 0.0  # Default no pruning
            return ratio_to_nm(ratio)
        
        return {'N': default_N, 'M': default_M}
    
    def ratio_to_nm(pruning_ratio):
        """Convert pruning ratio to N:M pattern"""
        # Density = 1 - pruning_ratio
        density = 1 - pruning_ratio

        # Map density to N:M patterns 
        # TODO: Need to explore more options of M values
        if density >= 0.8:   # ≤20% pruning
            return {'N': 4, 'M': 5}  # 80% dense
        elif density >= 0.75:  # ~25% pruning
            return {'N': 3, 'M': 4}  # 75% dense
        elif density >= 0.5:   # ~50% pruning
            return {'N': 2, 'M': 4}  # 50% dense
        elif density >= 0.4:   # ~60% pruning
            return {'N': 2, 'M': 5}  # 40% dense
        else:                  # High pruning
            return {'N': 1, 'M': 4}  # 25% dense
    
    def recursive_replace(module, prefix=''):
        """Recursively replace linear layers"""
        for name, child in module.named_children():
            # Build full name with prefix
            full_name = f"{prefix}.{name}" if prefix else name
            
            # Skip if in exclude list
            if full_name in exclude or name in exclude:
                continue
            
            if isinstance(child, nn.Linear):
                # Get sparsity config for this layer
                config = get_sparsity_config(full_name)
                
                # Prepare kwargs for new layer
                layer_kwargs = kwargs.copy()
                layer_kwargs.update(config)
                
                # Create new masked linear layer
                new_layer = new_class(
                    in_features=child.in_features,
                    out_features=child.out_features,
                    bias=child.bias is not None,
                    **layer_kwargs
                )
                
                # Copy weights and bias
                new_layer.weight.data = child.weight.data.clone()
                if child.bias is not None:
                    new_layer.bias.data = child.bias.data.clone()
                
                # Move to same device
                new_layer.to(child.weight.device)
                
                # Replace in parent module
                setattr(module, name, new_layer)
                
                print(f"Replaced {full_name}: {config}")
                
            else:
                # Recursively process children
                recursive_replace(child, full_name)
    
    # Start recursive replacement
    recursive_replace(model)
    return model

In [14]:
# model = copy.deepcopy(original_model)

In [15]:
# from maskllm.utils import replace_linear_with_
replace_linear_with_(model, MaskedLinearFrozen, exclude=[model.head], groups=groups)

Replaced blocks.0.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.0.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.0.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.0.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.1.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.1.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.1.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.1.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.2.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.2.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.2.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.2.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.3.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.3.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.3.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.3.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.4.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.4.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.4.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.4.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.5.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.5.attn.proj: {'N': 4, 'M': 5}
Replaced block

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): MaskedLinearFrozen(192, 576, bias=True, N=4, M=5)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): MaskedLinearFrozen(192, 192, bias=True, N=4, M=5)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): MaskedLinearFrozen(192, 768, bias=True, N=2, M=4)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (

In [16]:
count_parameters(model)

5526346

In [17]:
print_model_mac(model)

[📊] Total model MACs: 0.562G


In [18]:
# sparsegpt actual

import torch
import transformers
import torch.nn as nn 

def find_layers(module, layers=[nn.Linear], name=''):
    """
    Recursively find the layers of a certain type in a module.

    Args:
        module (nn.Module): PyTorch module.
        layers (list): List of layer types to find.
        name (str): Name of the module.

    Returns:
        dict: Dictionary of layers of the given type(s) within the module.
    """
    if isinstance(module, tuple(layers)):
        return {name: module}
    res = {}
    for name1, child in module.named_children():
        res.update(find_layers(
            child, layers=layers, name=name + '.' + name1 if name != '' else name1
        ))
    return res


@torch.no_grad()  
def prune_sparsegpt(model, loader, nsamples=128, batch_size=1, device=torch.device("cuda:0"),   
                   prune_n=0, prune_m=0, sparsity_ratio=0.0, disable_update=False, groups=None):  
    # Initialize input caches (unchanged)  
    model.to(device)
    layers = model.blocks  
    dtype = next(iter(model.parameters())).dtype  
    inps = torch.zeros(  
        (nsamples, model.num_prefix_tokens+model.patch_embed.num_patches, model.embed_dim), dtype=dtype, device=device  
    )  
    cache = {'i': 0}  
  
    class Catcher(nn.Module):  
        def __init__(self, module):  
            super().__init__()  
            self.module = module  
        def forward(self, inp, **kwargs):  
            inps[cache['i']] = inp  
            cache['i'] += 1
            raise ValueError  
    layers[0] = Catcher(layers[0])  
    for batch in loader:  
        if cache['i'] == nsamples: break  
        try:  
            model(batch[0].to(device))  
        except ValueError:  
            pass  
    layers[0] = layers[0].module  
    torch.cuda.empty_cache()  
  
    outs = torch.zeros_like(inps)  
    print('Ready.')  
      
    for i in range(len(layers)):  
        layer = layers[i]  
        inps, outs = inps.to(device), outs.to(device)  
  
        subset = find_layers(layer)  
  
        gpts = {}  
        for name in subset:  
            gpts[name] = SparseGPT(subset[name])  
  
        def add_batch(name):  
            def tmp(_, inp, out):  
                gpts[name].add_batch(inp[0].data, out.data)  
            return tmp  
  
        handles = []  
        for name in gpts:  
            handles.append(subset[name].register_forward_hook(add_batch(name)))  
  
        for j in range(nsamples):  
            outs[j] = layer(inps[j].unsqueeze(0))[0]  
        for h in handles:  
            h.remove()  
  
        for name in gpts:  
            print(i, name)  
            print('Pruning ...')  
              
            # Get layer-specific N:M pattern  
            layer_n, layer_m = get_layer_sparsity(name, groups, prune_n, prune_m)  
              
            gpts[name].fasterprune(sparsity_ratio, prunen=layer_n, prunem=layer_m,   
                                  percdamp=0.01, blocksize=128, disable_update=disable_update)  
            gpts[name].free()  
  
        for j in range(nsamples):
            outs[j] = layer(inps[j].unsqueeze(0))[0]  
  
        layers[i] = layer   
        torch.cuda.empty_cache()  
  
        inps, outs = outs, inps  
  
    torch.cuda.empty_cache()
  
  
def get_layer_sparsity(layer_name, groups, default_n, default_m):  
    """Get N:M pattern for a specific layer based on groups"""  
    if groups is None:  
        return default_n, default_m  
      
    # Extract layer type from name  
    name_parts = layer_name.split('.')  
      
    # Check if this is an attention layer  
    if 'attn' in name_parts:  
        if 'qkv' in name_parts or 'proj' in name_parts:  
            if 'attention_blocks' in groups:  
                ratio = groups['attention_blocks'].pruning_ratio  
                return ratio_to_nm(ratio)  
      
    # Check if this is an MLP layer  
    elif 'mlp' in name_parts:  
        if 'fc1' in name_parts or 'fc2' in name_parts:  
            if 'mlp_blocks' in groups:  
                ratio = groups['mlp_blocks'].pruning_ratio  
                return ratio_to_nm(ratio)  
      
    # Default  
    return default_n, default_m  
  
  
def ratio_to_nm(pruning_ratio):  
    """Convert pruning ratio to N:M pattern"""  
    density = 1 - pruning_ratio  
      
    if density >= 0.8:   # ≤20% pruning  
        return 4, 5  # 80% dense  
    elif density >= 0.75:  # ~25% pruning  
        return 3, 4  # 75% dense  
    elif density >= 0.5:   # ~50% pruning  
        return 2, 4  # 50% dense  
    elif density >= 0.4:   # ~60% pruning  
        return 2, 5  # 40% dense  
    else:                  # High pruning  
        return 1, 4  # 25% dense
    

In [20]:
prune_sparsegpt(model, val_loader, nsamples=128, batch_size=1, device=torch.device("cuda:0"), groups=groups)

Ready.
0 attn.qkv
Pruning ...
time 16.46
error 2542.26708984375
0 attn.proj
Pruning ...
time 0.05
error 176.33457946777344
0 mlp.fc1
Pruning ...
time 0.05
error 1119.800537109375
0 mlp.fc2
Pruning ...
time 0.23
error 47.417415618896484
1 attn.qkv
Pruning ...
time 0.05
error 8080.3271484375
1 attn.proj
Pruning ...
time 0.05
error 214.1728057861328
1 mlp.fc1
Pruning ...
time 0.05
error 2103.4521484375
1 mlp.fc2
Pruning ...
time 0.20
error 52.83512878417969
2 attn.qkv
Pruning ...
time 0.05
error 10999.73828125
2 attn.proj
Pruning ...
time 0.05
error 322.56982421875
2 mlp.fc1
Pruning ...
time 0.05
error 3235.1796875
2 mlp.fc2
Pruning ...
time 0.20
error 80.89598083496094
3 attn.qkv
Pruning ...
time 0.05
error 13087.875
3 attn.proj
Pruning ...
time 0.05
error 312.0545654296875
3 mlp.fc1
Pruning ...
time 0.05
error 4002.40478515625
3 mlp.fc2
Pruning ...
time 0.20
error 84.3326416015625
4 attn.qkv
Pruning ...
time 0.05
error 16457.578125
4 attn.proj
Pruning ...
time 0.05
error 325.14788818359

In [21]:
count_parameters(model)

5526346

In [22]:
print_model_mac(model)

[📊] Total model MACs: 0.562G


In [23]:
import os
model_name = 'deit_tiny_patch16_224.fb_in1k'

for name, m in model.named_modules():
    if hasattr(m, 'mask'):
        print(f"Layer {name} Sparsity: {1 - torch.sum(m.mask).item()/m.mask.numel()}")
    
os.makedirs(os.path.dirname(f"output/pruned/{model_name}_sparsegpt_cifar10.pt"), exist_ok=True)
print(model)
torch.save(model.state_dict(), f"output/pruned/{model_name}_sparsegpt_cifar10.pt")
print(f"Model saved to output/pruned/{model_name}_sparsegpt_cifar10.pt")

Layer blocks.0.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.0.attn.proj Sparsity: 0.7708333333333334
Layer blocks.0.mlp.fc1 Sparsity: 0.5
Layer blocks.0.mlp.fc2 Sparsity: 0.5
Layer blocks.1.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.1.attn.proj Sparsity: 0.7708333333333334
Layer blocks.1.mlp.fc1 Sparsity: 0.5
Layer blocks.1.mlp.fc2 Sparsity: 0.5
Layer blocks.2.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.2.attn.proj Sparsity: 0.7708333333333334
Layer blocks.2.mlp.fc1 Sparsity: 0.5
Layer blocks.2.mlp.fc2 Sparsity: 0.5
Layer blocks.3.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.3.attn.proj Sparsity: 0.7708333333333334
Layer blocks.3.mlp.fc1 Sparsity: 0.5
Layer blocks.3.mlp.fc2 Sparsity: 0.5
Layer blocks.4.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.4.attn.proj Sparsity: 0.7708333333333334
Layer blocks.4.mlp.fc1 Sparsity: 0.5
Layer blocks.4.mlp.fc2 Sparsity: 0.5
Layer blocks.5.attn.qkv Sparsity: 0.7708333333333334
Layer blocks.5.attn.proj Sparsity: 0.77083

# MaskLLM Training

In [1]:
import os  
import torch  
import torch.nn as nn  
from timm import utils  
from timm.data import create_dataset, create_loader, resolve_data_config  
from timm.models import create_model, safe_model_name  
from timm.optim import create_optimizer_v2  
from timm.scheduler import create_scheduler_v2  
from maskllm.utils import replace_linear_with_
from maskllm.maskllm import *

/work/hdd/bfxa/dshah13/mac_pruning_fv/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = {  
        'model': 'deit_tiny_patch16_224.fb_in1k',  
        # 'data_dir': 'data/imagenet',  
        'sparse_checkpoint': 'output/pruned/deit_tiny_patch16_224.fb_in1k_sparsegpt_cifar10.pt',  
        'sparsity_mode': 'maskllm',  
        'mask_only': True,  
        'batch_size': 8,  # increase according to compute available
        'epochs': 10,   # testing with 1 epoch
        'lr': 1e-3,  
        'weight_decay': 0.01,  
        'opt': 'adamw',  
        'sched': 'cosine',  
        'warmup_epochs': 0,  
        'min_lr': 1e-4,  
        'tau_range': [4, 0.05],  
        'scaling_range': [1e1, 1e2],  
        'prior_strength': 3,  
        'sparse_weight_reg': 1e-5,  
        'clip_grad': 2.0,  
        'mixup': 0.8,  
        'cutmix': 1.0,  
        'smoothing': 0.1,  
        'drop_path': 0.1,  
        'aa': 'rand-m8-inc1-mstd101',  
        'reprob': 0.3,  
        'remode': 'pixel',  
        'amp': True,  
        'output': 'output/maskllm_simplified',  
        'experiment': 'MaskLLM-Simplified'  
    }  

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
utils.random_seed(42, 0)

Using device: cuda


In [6]:
from utils.analysis_isomorphism import ViTIsomorphicAnalyzer

model = create_model(config['model'], pretrained=False, num_classes=10)
isomorphic_analyser = ViTIsomorphicAnalyzer(model)
groups = isomorphic_analyser.create_isomorphic_groups(
    target_macs=0.6*1e9,
    baseline_macs=1.25*1e9
    )

[🔍] Found 12 transformer blocks
[📊] Total model MACs: 1.259G
[📊] Target MACs: 0.600G
[📊] Target ratio: 0.520
[🔍] Found 24 MLP layers (3,538,944 total params)
[📊] MLP MAC fraction estimated: 0.666 (3,538,944/5,310,336 params)
[🔍] Found 24 attention layers (1,769,472 total params)
[📊] Attention MAC fraction estimated: 0.333 (1,769,472/5,310,336 params)
[🪫] Max feasible reduction under caps: 0.990 → best achievable ≈ 0.013G
[🎯] LLM suggests: mlp_pruning=0.400, attn_pruning=0.150
[🎯] Using as pruning ratios: r_mlp=0.400, r_attn=0.150
[🎯] Predicted achievement: 0.317 (target: 0.520)
[⚖️] Post-alloc solve -> r_mlp=0.400, r_attn=0.150 (mlp_frac=0.667, attn_frac=0.333, caps: 0.99/0.99)
[📊] MLP: 0.839G → 0.503G (ratio: 0.400)
[📊] Attention: 0.420G → 0.357G (ratio: 0.150)
[📊] Final group summary:
  - MLP blocks: 12 couples
  - Attention blocks: 12 couples
  - Output projections: 0 layers


In [5]:
model = replace_linear_with_(  
        model,   
        MaskedLinear,   
        exclude=[model.get_classifier()],
        groups=groups 
    )  
sparse_checkpoint = "output/pruned/deit_tiny_patch16_224.fb_in1k_sparsegpt_cifar10.pt"
print(f'Loading sparse checkpoint from {sparse_checkpoint}')  
checkpoint = torch.load(sparse_checkpoint, map_location='cpu')  
model.load_state_dict(checkpoint, strict=False)  

Replaced blocks.0.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.0.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.0.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.0.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.1.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.1.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.1.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.1.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.2.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.2.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.2.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.2.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.3.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.3.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.3.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.3.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.4.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.4.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.4.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.4.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.5.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.5.attn.proj: {'N': 4, 'M': 5}
Replaced block

_IncompatibleKeys(missing_keys=['blocks.0.attn.qkv.gate', 'blocks.0.attn.proj.gate', 'blocks.0.mlp.fc1.gate', 'blocks.0.mlp.fc2.gate', 'blocks.1.attn.qkv.gate', 'blocks.1.attn.proj.gate', 'blocks.1.mlp.fc1.gate', 'blocks.1.mlp.fc2.gate', 'blocks.2.attn.qkv.gate', 'blocks.2.attn.proj.gate', 'blocks.2.mlp.fc1.gate', 'blocks.2.mlp.fc2.gate', 'blocks.3.attn.qkv.gate', 'blocks.3.attn.proj.gate', 'blocks.3.mlp.fc1.gate', 'blocks.3.mlp.fc2.gate', 'blocks.4.attn.qkv.gate', 'blocks.4.attn.proj.gate', 'blocks.4.mlp.fc1.gate', 'blocks.4.mlp.fc2.gate', 'blocks.5.attn.qkv.gate', 'blocks.5.attn.proj.gate', 'blocks.5.mlp.fc1.gate', 'blocks.5.mlp.fc2.gate', 'blocks.6.attn.qkv.gate', 'blocks.6.attn.proj.gate', 'blocks.6.mlp.fc1.gate', 'blocks.6.mlp.fc2.gate', 'blocks.7.attn.qkv.gate', 'blocks.7.attn.proj.gate', 'blocks.7.mlp.fc1.gate', 'blocks.7.mlp.fc2.gate', 'blocks.8.attn.qkv.gate', 'blocks.8.attn.proj.gate', 'blocks.8.mlp.fc1.gate', 'blocks.8.mlp.fc2.gate', 'blocks.9.attn.qkv.gate', 'blocks.9.attn.

In [6]:
for name, module in model.named_modules():  
    if isinstance(module, MaskedLinear):  
        module.load_mask_prior(prior_strength=config['prior_strength'])  
      
# Freeze all parameters except gates  
if config['mask_only']:  
    print('Mask only mode - freezing all parameters except gates...')  
    for name, param in model.named_parameters():  
        if '.gate' not in name:  
            param.requires_grad = False  
    
model.to(device=device)  

initializing with prior (strength=3), Prior Sparsity: 0.7708
Block shape: torch.Size([22118, 5])
initializing with prior (strength=3), Prior Sparsity: 0.7708
Block shape: torch.Size([7372, 5])
initializing with prior (strength=3), Prior Sparsity: 0.5000
Block shape: torch.Size([36864, 6])
initializing with prior (strength=3), Prior Sparsity: 0.5000
Block shape: torch.Size([36864, 6])
initializing with prior (strength=3), Prior Sparsity: 0.7708
Block shape: torch.Size([22118, 5])
initializing with prior (strength=3), Prior Sparsity: 0.7708
Block shape: torch.Size([7372, 5])
initializing with prior (strength=3), Prior Sparsity: 0.5000
Block shape: torch.Size([36864, 6])
initializing with prior (strength=3), Prior Sparsity: 0.5000
Block shape: torch.Size([36864, 6])
initializing with prior (strength=3), Prior Sparsity: 0.7708
Block shape: torch.Size([22118, 5])
initializing with prior (strength=3), Prior Sparsity: 0.7708
Block shape: torch.Size([7372, 5])
initializing with prior (strength

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): MaskedLinear(192, 576, bias=True, N=4, M=5, tau=1, scaling=1, hard=False)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): MaskedLinear(192, 192, bias=True, N=4, M=5, tau=1, scaling=1, hard=False)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): MaskedLinear(192, 768, bias=True, N=2, M=4, tau=1, scaling=1, hard=False)
        (act): GELU(appr

In [7]:
from data.loaders import get_cifar10_loaders_pbench

batch_size = 1
loader_train, loader_eval = get_cifar10_loaders_pbench(batch_size, num_workers=16)

/work/hdd/bfxa/dshah13/mac_pruning_fv/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
/work/hdd/bfxa/dshah13/mac_pruning_fv/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [8]:
optimizer_config = {
    'opt': config.get('opt', 'adamw'),
    'lr': config.get('lr', 1e-3),
    'weight_decay': config.get('weight_decay', 0.01),
    # 'momentum' is not needed for AdamW but can be included for SGD
    # Add other optimizer-specific parameters if needed
    # 'filter_bias_and_bn': True,  # optional
    # 'layer_decay': None,  # optional
}

scheduler_config = {
    'sched': config.get('sched', 'cosine'),
    'num_epochs': config.get('epochs', 1),  # Note: 'epochs' in your config, not 'num_epochs'
    'warmup_lr': 1e-6,  # not in your config, using default
    'warmup_epochs': config.get('warmup_epochs', 0),
    'min_lr': config.get('min_lr', 1e-4),
    # 'decay_rate': 1.0,  # optional
    # 'decay_epochs': 1,  # optional
    # 'cooldown_epochs': 0,  # optional
}

optimizer = create_optimizer_v2(model, **optimizer_config)
lr_scheduler, num_epochs = create_scheduler_v2(optimizer, **scheduler_config)

In [9]:
 # Setup loss functions  
train_loss_fn = nn.CrossEntropyLoss(label_smoothing=config['smoothing']).to(device)  
validate_loss_fn = nn.CrossEntropyLoss().to(device)  

In [10]:
amp_autocast = torch.cuda.amp.autocast if config['amp'] else suppress
loss_scaler = utils.NativeScaler() if config['amp'] else None  

In [11]:
from contextlib import suppress
def train_one_epoch(epoch, model, loader, optimizer, loss_fn, config,   
                   amp_autocast=suppress, loss_scaler=None):  
    model.train()  
    for batch_idx, (input, target) in enumerate(loader):  
        input, target = input.to(device), target.to(device)  
          
        def _forward():  
            with amp_autocast():  
                output = model(input)  
                loss = loss_fn(output, target)  
                # Add sparse weight regularization  
                if config['sparse_weight_reg'] > 0:  
                    for m in model.modules():  
                        if isinstance(m, MaskedLinear):  
                            loss += -config['sparse_weight_reg'] * m.sparse_weight_reg()  
            return loss  
          
        if loss_scaler is not None:  
            loss_scaler(_forward(), optimizer, clip_grad=config['clip_grad'], parameters=model.parameters())  
        else:  
            loss = _forward()  
            loss.backward()  
            if config['clip_grad'] is not None:  
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['clip_grad'])  
            optimizer.step()  
          
        optimizer.zero_grad()  
  
def validate(model, loader, loss_fn, config, device, amp_autocast=suppress):  
    model.eval()  
    correct1 = correct5 = total = 0  
    loss_sum = 0  
      
    with torch.no_grad():  
        for input, target in loader:  
            input, target = input.to(device), target.to(device)  
            with amp_autocast():  
                output = model(input)  
                loss = loss_fn(output, target)  
              
            loss_sum += loss.item() * input.size(0)  
            total += input.size(0)  
              
            # Calculate accuracy  
            _, pred = output.topk(5, 1, True, True)  
            pred = pred.t()  
            correct = pred.eq(target.view(1, -1).expand_as(pred))  
            correct1 += correct[:1].reshape(-1).float().sum(0, keepdim=True)  
            correct5 += correct[:5].reshape(-1).float().sum(0, keepdim=True)  
      
    return {  
        'loss': loss_sum / total,  
        'top1': (correct1 / total).item() * 100,  
        'top5': (correct5 / total).item() * 100  
    }  

In [ ]:
for epoch in range(num_epochs):  
    # Update tau and scaling for MaskLLM  
    tau = config['tau_range'][0] + (config['tau_range'][1] - config['tau_range'][0]) * epoch / max(num_epochs - 1, 1)  
    scaling = config['scaling_range'][0] + (config['scaling_range'][1] - config['scaling_range'][0]) * epoch / max(num_epochs - 1, 1)  
        
    for m in model.modules():  
        if isinstance(m, MaskedLinear):  
            m.tau = tau  
            m.scaling = scaling  
        
    print(f'Epoch {epoch}: tau={tau}, scaling={scaling}')  
        
    # Train one epoch  
    train_one_epoch(epoch, model, loader_train, optimizer, train_loss_fn,   
                    config, amp_autocast=amp_autocast, loss_scaler=loss_scaler)  
        
    # Validate  
    eval_metrics = validate(model, loader_eval, validate_loss_fn, config,    
                            device=device, amp_autocast=amp_autocast)  
        
    print(f'Epoch {epoch}: Loss: {eval_metrics["loss"]}, Top-1: {eval_metrics["top1"]:.2f}%, Top-5: {eval_metrics["top5"]:.2f}%')  
        
    # Step scheduler  
    lr_scheduler.step(epoch + 1)  
      
# Save final model  
output_dir = utils.get_outdir(config['output'], config['experiment'])  
torch.save({  
    'model': model.state_dict(),  
    'config': config,  
}, os.path.join(output_dir, 'model_final.pth'))

Epoch 0: tau=4.0, scaling=10.0


/tmp/ipykernel_184346/2400490077.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp_autocast():
/tmp/ipykernel_184346/2400490077.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp_autocast():


Epoch 0: Top-1: 32.25%, Top-5: 86.14%
Epoch 1: tau=3.561111111111111, scaling=20.0
Epoch 1: Top-1: 73.45%, Top-5: 97.25%
Epoch 2: tau=3.1222222222222222, scaling=30.0
Epoch 2: Top-1: 83.84%, Top-5: 99.31%
Epoch 3: tau=2.683333333333333, scaling=40.0
Epoch 3: Top-1: 88.54%, Top-5: 99.52%
Epoch 4: tau=2.2444444444444445, scaling=50.0
Epoch 4: Top-1: 90.42%, Top-5: 99.53%
Epoch 5: tau=1.8055555555555554, scaling=60.0
Epoch 5: Top-1: 91.59%, Top-5: 99.66%
Epoch 6: tau=1.3666666666666663, scaling=70.0
Epoch 6: Top-1: 92.17%, Top-5: 99.47%
Epoch 7: tau=0.9277777777777776, scaling=80.0
Epoch 7: Top-1: 92.87%, Top-5: 99.32%
Epoch 8: tau=0.48888888888888893, scaling=90.0
Epoch 8: Top-1: 93.01%, Top-5: 99.32%
Epoch 9: tau=0.04999999999999938, scaling=100.0
Epoch 9: Top-1: 93.23%, Top-5: 99.30%


# Eval

In [7]:
checkpoint_path = 'output/maskllm_simplified/MaskLLM-Simplified/model_final.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu')  # or 'cuda'

# Extract components
model_state_dict = checkpoint['model']
config = checkpoint['config']

model = create_model(config['model'], pretrained=False, num_classes=10)
model = replace_linear_with_(  
        model,   
        MaskedLinear,   
        exclude=[model.get_classifier()],
        groups=groups 
    )  
model.load_state_dict(model_state_dict)
model.eval()  # Set to evaluation mode

print("Model loaded successfully!")
print(f"Config: {config}")


Replaced blocks.0.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.0.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.0.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.0.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.1.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.1.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.1.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.1.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.2.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.2.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.2.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.2.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.3.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.3.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.3.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.3.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.4.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.4.attn.proj: {'N': 4, 'M': 5}
Replaced blocks.4.mlp.fc1: {'N': 2, 'M': 4}
Replaced blocks.4.mlp.fc2: {'N': 2, 'M': 4}
Replaced blocks.5.attn.qkv: {'N': 4, 'M': 5}
Replaced blocks.5.attn.proj: {'N': 4, 'M': 5}
Replaced block

In [10]:
count_parameters(model)

12606082

In [11]:
print_model_mac(model)

[📊] Total model MACs: 0.562G


In [12]:
model

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): MaskedLinear(192, 576, bias=True, N=4, M=5, tau=1, scaling=1, hard=False)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): MaskedLinear(192, 192, bias=True, N=4, M=5, tau=1, scaling=1, hard=False)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): MaskedLinear(192, 768, bias=True, N=2, M=4, tau=1, scaling=1, hard=False)
        (act): GELU(appr